# Comparación de arquitecturas para un tutor de matemáticas

### Notebook 5 · Síntesis experimental y recomendación

**Proyecto:** Tutor inteligente de matemáticas · Comparación de arquitecturas
mediante fine-tuning
**Curso:** SI4006 · Tópicos Especiales y Aplicaciones en IA · Universidad EAFIT
**Notebook base:** `S04_Lab_Fine_tuning_Qwen.ipynb`

---

## 1 · Introducción

Este notebook no entrena nada. **Agrega** los resultados de los cuatro
notebooks anteriores y produce la comparación que da sentido al experimento.

| Notebook | Arquitectura | Modelo | Tarea |
|---|---|---|---|
| 1 · Qwen | Decoder-only | Qwen2.5-1.5B-Instruct | Generar la solución |
| 2 · BERT | Encoder-only | BETO (bert-base-spanish) | Clasificar el problema |
| 3 · Qwen+BERT | Pipeline modular | BETO → Qwen2.5+LoRA | Clasificar y luego generar |
| 4 · FLAN-T5 | Encoder-decoder | flan-t5-base | Ambas en una pasada |

### Por qué la comparación es válida

Las cuatro arquitecturas comparten, por construcción y no por casualidad:

- **El mismo corpus**, con el mismo hash SHA-256 verificado en cada notebook.
- **Las mismas particiones**: 132 entrenamiento / 33 validación, estratificadas,
  fijadas en el archivo de datos y no recalculadas por cada notebook.
- **Las mismas métricas**, definidas en una única celda compartida.
- **La misma semilla** (42) y decodificación greedy en toda evaluación.

Si alguna de esas condiciones se rompiera, las diferencias entre modelos
podrían venir de los datos y no de la arquitectura. La verificación de hashes de
la sección 2 comprueba justamente eso.

### Requisito

Haber ejecutado los notebooks 1 a 4, de modo que existan los cuatro archivos en
`resultados/`. El notebook funciona con resultados parciales, avisando de lo que
falta.

## 2 · Carga de resultados

In [ ]:
%pip install -q "transformers>=4.44" pandas matplotlib
print("Listo.")

### Persistencia entre notebooks

Los notebooks 3 y 5 **leen carpetas que producen los notebooks 1, 2 y 4**:

```
1 · Qwen    -> adaptadores/qwen-lora/      ─┐
2 · BERT    -> modelos/bert-clasificador/  ─┤-> el notebook 3 las lee
4 · FLAN-T5 -> adaptadores/flan-t5-lora/    │
1,2,3,4     -> resultados/*.json           ─┴-> el notebook 5 los lee
```

En Colab, `/content` se borra al desconectar el runtime, así que ese trabajo se
perdería entre sesiones. Montando Google Drive y trabajando desde una carpeta
suya, los artefactos sobreviven y cada notebook se puede ejecutar el día que se
pueda.

Con `USAR_DRIVE = False` todo queda en `/content`, lo cual es válido si
ejecutan los notebooks 1, 2 y 3 seguidos sin desconectar.

Fuera de Colab la celda no hace nada: el directorio de trabajo se queda como
está.

> **Los checkpoints intermedios nunca van a Drive.** El `Trainer` guarda en
> `output_dir` el modelo *más el estado del optimizador* en cada época. Para el
> notebook 2, que hace fine-tuning completo de BETO, eso son varios GB que
> además se escribirían por red. Como son desechables —lo que importa es el
> modelo final—, se mandan siempre al disco local del runtime mediante
> `DIR_CHECKPOINTS`.

In [ ]:
import os
from pathlib import Path

USAR_DRIVE    = True
CARPETA_DRIVE = "/content/drive/MyDrive/ProyectoIA"

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB and USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CARPETA_DRIVE).mkdir(parents=True, exist_ok=True)
    os.chdir(CARPETA_DRIVE)

# Checkpoints del Trainer: grandes y desechables -> siempre en disco local.
DIR_CHECKPOINTS = "/content/salidas" if EN_COLAB else "salidas"

print(f"En Colab           : {EN_COLAB}")
print(f"Directorio de trabajo: {Path.cwd()}")
print(f"Checkpoints en     : {DIR_CHECKPOINTS}")

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

DIR_RESULTADOS = Path("resultados")

ESPERADOS = {
    "qwen": "Notebook 1 · Qwen (decoder-only)",
    "bert": "Notebook 2 · BERT (encoder-only)",
    "pipeline_qwen_bert": "Notebook 3 · Pipeline modular",
    "flan_t5": "Notebook 4 · FLAN-T5 (encoder-decoder)",
}

R = {}
for clave, descripcion in ESPERADOS.items():
    ruta = DIR_RESULTADOS / f"{clave}.json"
    if ruta.exists():
        R[clave] = json.loads(ruta.read_text(encoding="utf-8"))
        print(f"  OK      {descripcion}")
    else:
        print(f"  FALTA   {descripcion}  ->  ejecuten ese notebook y vuelvan aquí")

if not R:
    raise FileNotFoundError("No hay ningún resultado en resultados/. Ejecuten los notebooks 1-4.")
print(f"\n{len(R)} de {len(ESPERADOS)} notebooks disponibles.")

In [ ]:
# Verificación de comparabilidad: los cuatro notebooks deben haber usado el
# mismo número de ejemplos de validación. Si no coincide, algo se ejecutó con
# datos distintos y las comparaciones de abajo no serían legítimas.
n_vals = {k: v.get("n_val") for k, v in R.items() if v.get("n_val")}
print("Ejemplos de validación por notebook:", n_vals)
if len(set(n_vals.values())) > 1:
    print("\nAVISO: los conjuntos de validación NO coinciden. "
          "Regeneren el dataset y re-ejecuten los notebooks antes de comparar.")
else:
    print("Conjuntos de validación consistentes: la comparación es válida.")

## 3 · Tabla comparativa

Diez dimensiones. Las que se pueden medir se leen de los JSON; las
cualitativas llevan una justificación explícita para que se puedan discutir en
lugar de aceptarse como dadas.

In [ ]:
def g(clave, *ruta, default=None):
    """Acceso seguro a un valor anidado del JSON de resultados."""
    nodo = R.get(clave)
    for paso in ruta:
        if not isinstance(nodo, dict) or paso not in nodo:
            return default
        nodo = nodo[paso]
    return nodo


def pct(x):
    return "—" if x is None else f"{x:.1%}"


def num(x, fmt="{:.0f}"):
    return "—" if x is None else fmt.format(x)


def mm(*claves):
    """Parámetros en millones, sumando varios modelos si hace falta."""
    total = sum(g(k, "parametros_totales", default=0) or 0 for k in claves)
    return f"{total/1e6:.0f}" if total else "—"


filas = {
    "Arquitectura": [
        "Decoder-only", "Encoder-only", "Encoder + Decoder (2 modelos)", "Encoder-decoder (1 modelo)",
    ],
    "Tipo de modelo": [
        "Autoregresivo", "Discriminativo", "Cascada modular", "Seq2seq",
    ],
    "Modelo base": [
        g("qwen", "modelo", default="—"),
        g("bert", "modelo", default="—"),
        "BETO → Qwen2.5+LoRA",
        g("flan_t5", "modelo", default="—"),
    ],
    "Parámetros totales (M)": [
        mm("qwen"), mm("bert"), mm("qwen", "bert"), mm("flan_t5"),
    ],
    "Tokenización": [
        f"BPE · {g('qwen','tokenizador','vocabulario',default='?')} tokens",
        f"WordPiece · {g('bert','tokenizador','vocabulario',default='?')} tokens",
        "Dos tokenizadores distintos",
        f"SentencePiece · {g('flan_t5','tokenizador','vocabulario',default='?')} tokens",
    ],
    "Método de fine-tuning": [
        g("qwen", "metodo", default="—"),
        g("bert", "metodo", default="—"),
        "Ninguno (composición)",
        g("flan_t5", "metodo", default="—"),
    ],
    "Tiempo de entrenamiento (s)": [
        num(g("qwen", "tiempo_entrenamiento_s"), "{:.0f}"),
        num(g("bert", "tiempo_entrenamiento_s"), "{:.0f}"),
        "0 (reutiliza 1 y 2)",
        num(g("flan_t5", "tiempo_entrenamiento_s"), "{:.0f}"),
    ],
    "Capacidad de clasificación": [
        "No aplica",
        pct(g("bert", "finetuned", "accuracy")),
        pct(g("pipeline_qwen_bert", "accuracy_clasificador")),
        pct(g("flan_t5", "clasificacion_finetuned", "accuracy")),
    ],
    "Capacidad generativa (exactitud)": [
        pct(g("qwen", "finetuned", "exactitud")),
        "No aplica",
        pct(g("pipeline_qwen_bert", "config_B_pipeline", "exactitud")),
        pct(g("flan_t5", "finetuned", "exactitud")),
    ],
    "Formato válido": [
        pct(g("qwen", "finetuned", "formato_valido")),
        "No aplica",
        pct(g("pipeline_qwen_bert", "config_B_pipeline", "formato_valido")),
        pct(g("flan_t5", "finetuned", "formato_valido")),
    ],
}

tabla = pd.DataFrame(filas, index=["Qwen", "BERT", "Qwen+BERT", "FLAN-T5"]).T
pd.set_option("display.max_colwidth", 42)
print(tabla.to_string())

### Dimensiones cualitativas

Estas no salen de un JSON: son juicios de ingeniería sobre lo observado
durante el laboratorio. Cada una lleva su razón, para que sea discutible.

In [ ]:
cualitativas = pd.DataFrame({
    "Facilidad de fine-tuning": {
        "Qwen": "Media — LoRA obligatorio; los parámetros entrenables deben forzarse a fp32 o la pérdida diverge a NaN",
        "BERT": "Alta — fine-tuning completo directo, receta estándar, sin sorpresas",
        "Qwen+BERT": "Media — no entrena, pero exige coordinar versiones y alineación de etiquetas entre dos artefactos",
        "FLAN-T5": "Media — LoRA sencillo, pero fp16 rompe el modelo: obliga a fp32",
    },
    "Velocidad de entrenamiento": {
        "Qwen": "Lenta — 1.5B parámetros, secuencias largas",
        "BERT": "Muy rápida — 110M, secuencias de 64 tokens, segundos por época",
        "Qwen+BERT": "N/A — hereda el costo de 1 y 2",
        "FLAN-T5": "Media — 250M pero en fp32 y con dos pilas (encoder y decoder)",
    },
    "Consumo de recursos (inferencia)": {
        "Qwen": "Alto — ~3 GB en fp16 y generación token a token",
        "BERT": "Muy bajo — ~0.4 GB, una sola pasada",
        "Qwen+BERT": "Alto — suma ambos, dominado por Qwen",
        "FLAN-T5": "Medio — ~1 GB, generación más corta",
    },
    "Interpretabilidad": {
        "Qwen": "Baja — solo se ve el texto final; no hay estado intermedio inspeccionable",
        "BERT": "Alta — distribución sobre 11 clases, matriz de confusión, confianza calibrable",
        "Qwen+BERT": "La más alta — el estado intermedio es explícito y se puede atribuir el fallo a una etapa concreta",
        "FLAN-T5": "Media — la línea 'Categoria:' expone la decisión interna, pero no es un estado separable",
    },
    "Escalabilidad": {
        "Qwen": "Alta — más datos mejoran directamente; hay variantes de 3B, 7B, 14B con el mismo código",
        "BERT": "Media — añadir categorías exige reetiquetar y reentrenar; el espacio de clases es cerrado",
        "Qwen+BERT": "Media — escala, pero cada componente por separado y hay que revalidar la interfaz",
        "FLAN-T5": "Media — mT5 resolvería el tokenizador; flan-t5-large necesita más GPU",
    },
}).T
print(cualitativas.to_string())

## 4 · Visualización comparativa

In [ ]:
import matplotlib.pyplot as plt

etiquetas, exact_base, exact_ft = [], [], []
for clave, nombre, ruta_base, ruta_ft in [
    ("qwen", "Qwen", ("baseline", "exactitud"), ("finetuned", "exactitud")),
    ("pipeline_qwen_bert", "Pipeline", ("config_A_solo_qwen", "exactitud"), ("config_B_pipeline", "exactitud")),
    ("flan_t5", "FLAN-T5", ("baseline", "exactitud"), ("finetuned", "exactitud")),
]:
    if clave in R:
        etiquetas.append(nombre)
        exact_base.append(g(clave, *ruta_base, default=0))
        exact_ft.append(g(clave, *ruta_ft, default=0))

if etiquetas:
    x = np.arange(len(etiquetas)); ancho = 0.38
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(x - ancho/2, exact_base, ancho, label="Antes (baseline / sin enrutar)")
    ax.bar(x + ancho/2, exact_ft, ancho, label="Después (fine-tuned / pipeline)")
    for i, (b, f) in enumerate(zip(exact_base, exact_ft)):
        ax.text(i - ancho/2, b + 0.01, f"{b:.0%}", ha="center", fontsize=9)
        ax.text(i + ancho/2, f + 0.01, f"{f:.0%}", ha="center", fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(etiquetas)
    ax.set_ylabel("Exactitud de la respuesta final"); ax.set_ylim(0, 1.05)
    ax.set_title(f"Capacidad generativa · {list(n_vals.values())[0]} ejemplos de validación")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

In [ ]:
clf_etiquetas, clf_valores = [], []
if "bert" in R:
    clf_etiquetas.append("BETO\n(especialista)")
    clf_valores.append(g("bert", "finetuned", "accuracy", default=0))
if "bert" in R:
    clf_etiquetas.append("TF-IDF\n(baseline clásico)")
    clf_valores.append(g("bert", "baseline_tfidf", "accuracy", default=0))
if "flan_t5" in R:
    clf_etiquetas.append("FLAN-T5\n(implícita)")
    clf_valores.append(g("flan_t5", "clasificacion_finetuned", "accuracy", default=0))

if clf_etiquetas:
    fig, ax = plt.subplots(figsize=(7, 4.2))
    barras = ax.bar(clf_etiquetas, clf_valores, color=["#3b7dd8", "#9aa5b1", "#d88b3b"][:len(clf_valores)])
    ax.axhline(1/11, ls=":", c="red", lw=1, label="Azar (1/11)")
    for b, v in zip(barras, clf_valores):
        ax.text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.0%}", ha="center", fontsize=10)
    ax.set_ylim(0, 1.05); ax.set_ylabel("Accuracy de clasificación")
    ax.set_title("Capacidad de clasificación · quién identifica mejor el tipo de problema")
    ax.legend(); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout(); plt.show()

## 5 · Análisis cualitativo I — Tokenización

Los tres tokenizadores se comparan **sobre el mismo texto**, cargándolos aquí
directamente. No hace falta GPU: un tokenizador son unos pocos MB.

Qué mide cada columna:

- **Fertilidad** = tokens por palabra. Más bajo es mejor: significa que el
  vocabulario captura unidades lingüísticas del español en lugar de trocearlas.
  Impacta en el costo de cómputo y en cuánto cabe en la ventana de contexto.
- **Tasa de `<unk>`** = proporción de tokens que el modelo no puede
  representar. Es información destruida antes de llegar al modelo, y ningún
  entrenamiento la recupera.
- **Tokens por ejemplo** = costo efectivo de procesar un problema completo.

In [ ]:
from transformers import AutoTokenizer

MODELOS_TOK = {
    "Qwen2.5 (BPE)": "Qwen/Qwen2.5-1.5B-Instruct",
    "BETO (WordPiece)": "dccuchile/bert-base-spanish-wwm-cased",
    "FLAN-T5 (SentencePiece)": "google/flan-t5-base",
}

tokenizadores = {}
for nombre, mid in MODELOS_TOK.items():
    try:
        tokenizadores[nombre] = AutoTokenizer.from_pretrained(mid)
        print(f"  OK  {nombre}")
    except Exception as e:
        print(f"  Falló {nombre}: {e}")

### 5.1 · Materialización del corpus

El corpus vive en `data/math_tutor_dataset.jsonl`, generado por
`scripts/dataset_fuente.py`. Para que el notebook funcione en Colab sin subir
archivos, abajo va una **copia comprimida** de ese mismo archivo.

La celda **no sobrescribe** el JSONL si ya existe: eso permite escalar el
dataset (reemplazar el archivo por uno mayor) sin tocar el notebook. El hash
SHA-256 que se imprime debe ser idéntico en los cinco notebooks; si difiere,
alguno está entrenando con datos distintos y la comparación no sería válida.

Hash esperado de la versión embebida: `cf4732834d64a196…`

In [ ]:
import base64, gzip, hashlib, json
from pathlib import Path

RUTA_DATOS = Path("data/math_tutor_dataset.jsonl")
SHA_ESPERADO = "cf4732834d64a196d49eda88ddac8694a529a8bfc6b843e8eb8d74b8c6d59ecc"

_BLOB = (
    "H4sIAAAAAAAC/9V9XY/jSJLYuwH/B2KABrrR3ZL4KaqAQWM8O4DvMLc3t+Pzi20sWBK7mmNJrKWkQlUfDPje7h8s4Ld5nId5"
    "2BsbBublgKkf4P+wv8QRkUkyM5lBJlWqKhWw21OimBmhjMiMz4z4py+K1RcX3he7w+b9zP/iHfx1vS72+GhfZcUWnyyzfX5V"
    "VkUmX8y0h3+kCWbwaF9cl/hKeZ1X2bIoafC2uMnX+PQy2xXLEh/BqKt8K5/l+CTfArAVzf+HfHfI1ze5t868XXF1KOC73JNT"
    "3v+yvfCCKPbeeuE8nRC62boQI7/LdqXnX3jfA4abcucdtvDFKt+981b5Mt9mO+/OW+Js+CeM2ubZdlUCnJ23LD5W2W7yX7c0"
    "R6DA8L70kiCEbwCv60O+22fex2KbrS/wMcK/hhE7AP9fvjgeLs5jQvziv8HDqgb6RwKKv1KCvcnWZaV8znd/XOUbXP6P2XqX"
    "/49//+/+SaVs8CIoG07muAjBJOJo+xUtH64yTLIsNxmucbHJ1l4Gk+3gk4c/o1JIqUyKSzthqDkxyekMCodZgLAEnJgUnDiR"
    "MDwNCa+r8nKdi9fGUvBvD9nWq/JlWVVAMFjZ2Pvvxfr+l02+r2CdrssK12qT3f8l22bA9cEktb2wz6pVPvF++7evD/c/bvfw"
    "hfpSO3++9fblPlt/sPPCN2vxtZcTifC3I1FwX61gnlUBy75dFtq+RpTfEl5IJYYTQhUfnSncYQqu0KHxLNEBqXCHC3NET72/"
    "t4f8phy9wf0gnuEpF6WzgcN7XYoV3d7/uslhTRQaKpPAukbzcGYjIz63n8+dqfE1y6QcteqZGxLVDwZoFL8MGs3EUTabJAyJ"
    "vtktq+KywKWEl/BoLPGPmXedVZkHc2brrBKrLA7MXKVeOz0utD8JYxv18Lmx846AijNY4HGErYE2hK0fDBA2eSmbDw8j0jJS"
    "x913XRW0RdrBX4KKolCzHpDDuZhXS3gXztg5gYBX51bazuP+fckAxUGD4DjSznXCzl3IOn9igWuj6j+CHL0sLtdFuc+XIDnz"
    "7f3PmeeDrrguLlFa3qG8hH0B4jJd1A9ppp0mYeU3e2CL3Ms+lVX2gdOw5IyVPhutOYm+7LABDLN3Xra7/9n700HoQRv9jCZt"
    "FjD60gvCyMYF8FgCMNWu4+HTQa5D5hhCB98wBjx24IxU44wbWsO93LhPyh5fZ5sC9jvSswIO8OPZDA7EfCeWa33Y5sghAcg2"
    "5THorKC/KtzRjB9WuOjN0q4Dye27Ag7VWIGQeiuxwK0M/7VxBH0h0OwoXc5wiQWsEDlOMMA2rEDPHZhhcS7M8BVYnh7+tlVR"
    "evS8Kreen4Jik60O6z2dFsDgM29b3P9FPR9AewU5sSvRQG0GDjDD3x5gnHpsX1WHa5K+zVwqE6SkXhF0UK/8hV1ng+fNaJ0H"
    "XMAR7buAWD3OhNYqdPDNMOX92WPK/QLEd7XJgZankP3RbJHAsiyC9Hi9u50DVjaeBdZjHZ+PULu7c3LUqiduiFQ/GCCSfyZS"
    "vDrs6bja5dtVXhU7UF+FMIZNKzQeD96Hp/BOMAl0y/oO39rlVwd0HDWvkdNBea3e0PUJuS63V8X+sJLHJtmphMeHAR5QhioH"
    "LMHVnGQToU0iGmjg2oxp3pR2g0auMQMOa0bzRrQDowQvZzf76WQBCzKfzOJhG41eJnMJ/3K00sSrAgRqUvFkYdXl6QvWUBsB"
    "WpzdFqCsAldDbnW3+skAocMzORFw+68y7wb+RfU9mntLUOd2OYpjUNN29z9eZrDd77wkxM+rclNsr0pNarcD6llWhwrdkzgA"
    "lkocOBvwxg1ueZoCZ+pR5SI0tQAbsKN9qwCHxy1O3HbnIZFE0GCwproJqLXZfRfRHZ1QaXuKHT9PyawJ/Dg+XoArk+DygpJs"
    "JyJ+MUKG26Zl6VbP3dKrfjJAsfgMdu03W5DDYJCXq/wq8z5ldyClctyFPwhuho32OUMleomKdwBwFzHu31kTE9J2b/M6zjSg"
    "bqvbp3WSZBI4ukrw5FzQuenHFg8N+t3JZZLRkAt8DXcaqsmB3VEDjxsc+c3sgI3quhnCgz3vDWTaY3/ImQOTuYU2K4Ro4R//"
    "6SJgKbi83ntRvGB2+R8QRVxGWxwRWO26WKFUKYEiZAmCnliCvgkcBgwLC7PPFNZogKGNHFuDIfBYp/wJEMAJTdCscR7rsTL5"
    "eYDYwQsh9iSCRQgnwXxYi8N3SZOCPxx1OHpTzI/G1cS3UhifswqcM1SiaRcea8xJoK0xJx8MEDZ0Fdsu1LWKAk5sWwj8LZ5k"
    "G6TnHpQuD080YPQEPBMZnolf5+tdcbBERUEMLKW2tqhfrtU2eHFbLj/lrSn38ZCLM7OFRMvIyIl/3Mr54eMuB1cq7kNUtzxa"
    "EtiZCockQC7cge/Dd14uyFm9EzJN4nWZ/VB66Ge3sc770PitOh85oIIDxiDB8ZMVk4a53ruwVvTkZ8ZxmiD5Nt+DhJ3HQzKi"
    "RxVUZiEHaRDbPbJBzBz/Vk3QMivvdQ1iw90auMjx+IWQyaeVgLCVi4UO7worGTRnVwOdXhUQKGNoEtuTlCY95rkzXLLOLRD5"
    "JKVJbGQpTVyom5yIuieNuZGXLgSNtgm5Cd0Gg1y2GBtEpFaQJwO5INflFubJd8xZ/W07ptaWdnhS4ql5wIMSD1AZ9VJzWALa"
    "ZBFZW3bNHR5bg2ujIFIGiwGLtes0gK1x56Sbz8+D6pCXtMVoIhhvguaYEgJIVSLMkMHOANrDah32xboAQ8RLU/m9wQb7lg0+"
    "DJzRsNo0sZwT/EHgqUUMcrAYS6/YFpBNpFIfkXqPkIEgsKtsxE8CiQNzdjvCJNerDo0lvwayJX8SOJA/fdQj/cSe15kQcUn4"
    "IPGrToOSMmHkbzJO/tqmZQVwYgrgxEkAL85hs361XVX3P+3qfIhIiXvfeVfZbg+6dZA2cV011r0CsxUiyetcbFA+CwKngfdK"
    "oa1u6BBY18O7+zKSyo8Ai9uFCXPTF7YwtytM8pJaofHb0x7ipufDNDcCnQ80u57EXTpB98JskjpoX/guaUHwh6PyRW+K+UVm"
    "Z8wkkvK6lzNU8rB24fWkksZm/qjDtvb9x9zWY4j7ffkRt/SmAJHrT5LUa0Kgu4P3CabJtmX9bRx49sjnqviYVzkm4qLkRrSF"
    "iQ5gMDQCK8ypZNpYkPXlpdAFapMVZ2/mgzwkMO4zYd7flWrWOaH+XuD4JVDOt4ZH8blnC44+EA8SBl0MOKYx0Gh4B5878E7w"
    "QmyyZIb+NvbeiIvgrqdAB1dgJSk8HiOyjQlZh5mYtfWXBS6EeXx32RjtOs/2VZ30gPIIhuZbkZUExj9yM0hor1wertEs0VTq"
    "5lVpW6GpwZpV2oI3I+t5Udc1LapYGuu4UXxIkbFJbchQqefqoW8POOkg0QCxAtuA1krsIHWge/SCNOr3MXmMmD35HeQqFkIP"
    "wjcx2xWYaJnjr99hfnAdWMP4GWa35Eug2fawuf+pKpZqsEPCQUejP7d6M/25TtiHwsbZdKis+1KAbh2W4vMAleNzEdkQFv0P"
    "5VUJ29f0VItrUn5k+sQp3Ama7e/zGwoRwZ9/OBSf5fsdb26/eFcAOkn1fGeiSUKTpCh9ReJUS2YGIr5+H77BXRthJhJ5Q+yO"
    "l16X+FGIiKxmOwq8P6bPIT7EXpvD2i1iCi8CF62B25emWBF8Fjzh5cHEu/+zhxmEVh74Hbi4yg145HD3tm+TmUx/0vUf+jNS"
    "VX167kf4YppYUx7hsU7lUaBIvzeAcFSVkBo6ys8DhAyehpAjImcOqhr4mGF5/AknGP6uQRgPY7KkdgXEzrxPh8uCkgvgxiYs"
    "fiLmwRN4ns4Uwn6D9wx2MAseQEIpwe0iA9mN/YXGHl0Jg/9YL7NMUp34RyKGkxyDEnvhReDV3niZpA6sEp6aVR4aYwW9MTvs"
    "y839LzfF2rspIGME7BxMwoG5yiXKX/Rcok20Rzc27NfOjVO87DJw2zSvQHaBSAroZdJIZ2BkbgE0p2V+XW5vcqkgoMu8gGMc"
    "blKICS6sM2GGC/A1faFw4u/qS6PwffurgDnknF/ij8Jjg+Iufmr3KgFHsKm2D8UVZxuPJSubuqi2kimdObCpc/bew3nVUTx9"
    "l68qeY8mw+ynxh25RPe6SMG0eCE/ZZfIeWIcKUKwWujYBE5O5TDezLnO93BLCm/0UDq5vCNzk8GvwxzLJWqmkGXWrIGm0xCO"
    "QC+0PBcJkwpIX9h8lccAF9dsO2A5NjFgNyxCzx2YJH4+/eXoy9G0Mk4Sr/9+dL3Cfk+KZ5/Y4q5Ia/P25Xh2UjwdCJa8NIJB"
    "qjKsRzhzIph8mfR3H45aTAdaFaJeSJ1ho+RZ08TiZTsFI7+XgsPgZLK1BoglqYTWkjTyXUg6fxJ9YoQTKltvsuX9T1txxxRM"
    "01hm796Jkxo/4NWTHPX19f2P18UyN6K78mFz7/aDC/XhnJR7CY92AZISw9QZ6ZsVqTfa/UnFfLBfnITk2XqSPp4YjYS4TKmB"
    "58OJOg5KWNGFUdJHkehP4b8KhKUSTSKnY2AHlzKkBSCOUzL6fH92lG3i+xMR/vPttzH6uKEHk1HGiIIDf1vDuKLhwBGL59Xx"
    "bMzwrYggUcIEHqObwy6HxESx5m20mfaUvFeq6nzXkAK9hZzL+sqp68FxjSdNSbPqO9hyz5awAFomGE/gpD99YdPqjgBNib8d"
    "oGwYw7frdLGbiuB04/a8dLqZNPPdvFLN66Rjib/JNST+DrQUEnHR2ae9F9glAz7v9U0NAhRpJSYoPm6RGHpC4HL8+/7Z6QmH"
    "VbGHt+qjbw4rum7VBPwgv/HBTivW8J12tUc8crnY0yeiBUyRQN7MWQto/FJ1VBLtUkEhhhfkFO46wjAC5L7UQPfwhgJf5RAH"
    "BnlmH+axlZSEYE041YBeyP90KGA1ciFTlgeIOpVKbZNVcVMIxT2JZAJDpJVTqmFQFpA94Ugn+NFQRTklHR6fcmRkGjkQ+Xm9"
    "j8xJcHlAR8ayxMwc6SwEKav5EGk34A3WIS8jngUQvaHLrq7HgSyHh86hIquswti8ppuKc1yocwvmou6ix194PBJCHdDB81rh"
    "oscROFiCg1gmeonnAtYmQeEaO4aq6tfRBJN/i1p19Ld62TNJqfCNPyeNPGUyENN41q8VDEKkzJUuLDZuFRseoTR2UvfiMzwQ"
    "tHv7OdxcuizXOyEVM/CAfhb1O+iAKK8qEQSq7WnjeKi/l3f3j1AQJGy5G1GvZpwIi9Y1HzBXRQIda3clYQAJnMgEz17y7eLQ"
    "3vMdvEICQsstbI3SbWfnlvBEAYG/yypMYxQ6IhSFBQd4tsnXlO0EohdSAPBCHGaYVGtpMlL+pxS2iQfFDa6MLKh2DlkhDUIH"
    "tMz0Lh8mICgoe66p8pcAJPzzQs6rgQHA9f7/kilnjXcqv6QTEXCAQ05HBQJ7Yuhg2oPDgQOC03HAY18ABi0KloK3DhstrMfd"
    "30xCIThrjDowTnt+WpLf+oRshDnQA8yBA2nCxybNSZMQoHJBgisxc/TtZZtLuZV9mdINOx2KHVP1Flk7WuTzwy8sKwRA80ea"
    "t1c8Qsm7sDp7F30n8/EoCF+vCpx19S50B+/CgfDRI5/K41IKQJvPr0CLLSvwyMENKEooyG+Xh2qHlBcLB4XJ4HeDH4DKfU+8"
    "r/GkRWtAnOnL7DoTMXB6GxI2tbeVYxvG5HTLbotR0QIv2dEQwGgLVwQrJA98KldsrkGzYTEiixSicmnhJAq9DJbjFsi6yrAo"
    "Zq5w0u/p1jUQD4TxYQcw8EeC83ZJ3E2C+7BTTL4qp0oNxEafMgz0gz5TXEKFjkj8BGvFPvEVc74Mo4sDT48oW/Gvxbat9efA"
    "vfF56BQuSQaivkCO2aiQnlqJ1zrXoNCZI5ILGqb8MCiCMB9x316iqEP+YqK1mVswEzJkJmIDTBS6xoKVTwMwm5QCExofm1ZB"
    "KhFqBy5InkOvOP6aIm28+CGaRTsLltK0V+ycjVItzBnZjWoUWHchz9w1YHTqncpWimrkDLqBUbYqEkLULa6NgJzKrUZ1UVOp"
    "OGtSxBiLtyBgCO18GjW4f5v6yc1mUmw4AXicHqKixHGBC9AxmocJc5wWkj6X+nnsLcaYNgtbQ4LX/bDKujwAlBBB3Ow9e7En"
    "V81SmZ38/+q8LO10cjlQa3FOG/p7YN4NaHyw7QLwgl1l67Uo1AgeXQjpZrAlFZNXk7TNu/Cfm0xsWTngYZu2hqr1LRKSEK1r"
    "314aIm4QOmrTNkBF0yINHCt1dZit2DUcOvvqYHKBU8j3TKTuIqKNMH+IzK3nwAVlWmOYt4oHZK4xI0sjs/dF6OBt85/D28b5"
    "ZT9iMXywBuT9/zCeKdXwtx6UDwTFGAwJVCohZxV3IhT6oAxy3JIbvSa+nOWa7uPsXDaquLzP7xwCr9ZxiWvFFU/OOVeOcz5j"
    "sjWOAE4lXSxgWR/PzJ6vMR8u2knsEbwsgeuLdUkng2Y4+icxfTCoU523NBguqZeXYNZrLh0tkyq/zZZAqBwvENr98H6nZNMo"
    "uJ3cLQMiH543Sjb5Tv529zvGj+HeYSS1cLCjgnyToUMhjZQ6Pj8crrA9AoQt9nmdz4CNusxKPvBN8woo6iC+tmw5H20r3pRr"
    "7FCibUZsIdc4jOroiJxe9eWS5zUUgRJ725SowarnNBiNAvl9deBsjEbHoI3POHhS/OhlHQhRmtCJsGDblRUb+DEfpY5MvlYs"
    "ziBzrJHNSG+mmWgiLdIiHmHWnL2XgtlJYTQwEW5RwbCZeUZzBRdixs/k1GUyNWCOjyCpqjq+9rGEYiEVZbrITiG6BY5Fmin3"
    "pRIRyzhkXbj1HBgkBzVWFE2j+bfDLluYt453EUxY/WJ1KLXA/fflmjJJqQTb2gZRbXcEk6TNG3e1OyDWfh2m94pX7GE8OZyV"
    "NDzWVM35MfHlg4IK0q4hQUDILSjcnCcwFwy6BKgrccQZfBs9XYDQp8YX6RyLRvSUhC5hElmooPhM3E8pytBnJF9+ypRpsBy3"
    "mkRa167AKtwBAaFr5aH9anu3FPRIwDieBclfZQ+Ny+uhA8WDp6X4SeOOGCD/s0dl0cGlPWFrGqipLxgpEYEpqMUO8EHIKxNh"
    "IHfWLQrfXG8PKIXH7uJNTR/vEZCVAvBdmOx2N7zA6ZAbmAjvrI+e435/HRLR0zdtbYDeDS9bJVLeDSSc/IQnaYEJXGIaFPmh"
    "QnbNoyeO+D+Tz1WcCl1l0w/YLe8EGkf3AmWVTF/PLpCfB2gfvdBjfiG0TCzXxUZsqHQI6lLW/dbM8CUdGbYT3qcacuSltffy"
    "MRv5jIKoH+06LPaGr25wDjXvIRLHL1WSR8j2r0nkxW+O39liAir+wO5rAQuF7sKaw7VIR+9qBaxlT2sAWWrrBSkWLnpb8pIP"
    "c1Hsy5+LfqTHa2/tPBBfCWKbGA+Dpulp5Ef21orRaO3NBKxJcQMk32QxMlorRg50n79g7e019nsHETiZkxQPhwlvbrZmAlzb"
    "Scxv9EjcPaVb3iHjTQwnMUv4IcCWrW6C5MMJpjsxnLic7+kLpryPq5eIo7Cnpp1Ff4awoDx5L+oJgAKGpTZobCH4CLM2k5no"
    "OiVLz0VWKRClg2o9h5YU9g9BiD0vdCkRuUiJxTNIidMeGRGWCVy8ER6Xt3SVeuyhIaagW/pabaHGoRMmtUOneSFURIiA21PY"
    "zvkUUTERlYOsOGjSRIXOOwJGFbFD1nCKHp8rV5Ch9Br3UPyGmjCMZwoaLI5qVogQGD+sE31qxmiVefDQvBfXuQKmB6gzayj4"
    "WGSLholuUGg48C1BjX6gDgziv2RNQyTqC2/Bax+33dzBtJAJKApZ5Dzk68VMNZxIXgBhmCatKwHZSyImvLI5ArqFRVS4bAl0"
    "XeNMHBRO/9HchQ/J95XF7y7vf90pyeJ4iMLKZj9Qhof3DdWelQ0YKbEcw3pQ2wV97k1Xd6yefolJEFpYo5lGySjEJFsJ1KW+"
    "Mvr3CRod7u+JNkFq6z8pXyY8LjAlXBwFoT0VLWyR6ym3zEDX+k3ycNlUNQN4m7Xm4Hl2usX8DKwkaxu1zW3U+kbfF1pfm0Vb"
    "DujOWx8AIfw6u6pyrGACKhxF4CDE89u/LTt1kTDZFdO0M1kWGcG6cFIzAdwBWFF8IJJ6hL2fKeY5ERoXtaucOojaG5ky9ZHc"
    "ETBZygqa711qr40UuKgw0ZPy08i6z1gFB7q5Yp7JLqfgH2ahhFhQAMyArTAOoli/qXDYYlfroO5EhYHZWecNNRcLr4dkV5CN"
    "NVBL4+saqtQlItkcJYy18krfElyse4mFDOSdgWDWvhN22ylTftXuQswlwhc0LoiZS9b0hS1tqxdJfJVHz2TCYcR4VcneEYae"
    "OzBlfB5q05F51QEqHK8joVkf56OJpBhZ2I0tAWKhKlGhXYkKe1J93E2uBhvD4FLx4NSpcCjfZ1Saz3W5d4u3w4tUBn33R/ha"
    "nowGY8RP6Kz97V+xwMFv/8qwwtfZenlYN02vG+TFSNxuWD6X/lzYPLTkLSWj2Err0KD1KHCaJa0CYhUcnaShC0mDF0fS4Ldf"
    "+sKo2hI3KyyGDds+dFPFGjQNgx5SWuBw21JCYGmoB0jDwIGG4Yuj4V//5X9hWfy3bE4ueC2r7P7nz6RxkO1DDScike+K6duo"
    "/PtBW+Usimz7E757K8wHuzTv5kCMB6vtUxUge/TqCRDBzIHG0UukcRCg1Q9/RAu38xcXf4nVJ+Vg1NHhRKQZjOCoEpVWHRvd"
    "PJfeE3gQoBEAV7wYXHrLmCw2Im38VKQ9rZdqAWIKkwHeiGsob6Xv1/1EXpCcS317OKTjY/Tr1INYdXS3+dHoC5GbUNw6nEvX"
    "kwjf4F61V1x2O9cbbI0oSS+emtrmhCFfqtko1OzAWMlznhlHZmb/9gvX2hFv2VXU0vZwWRpF6NVbMdRinq5RoAJNEtdo/hLV"
    "chguuN3hP/Kjm9PzwYhQ9jaPwon8n8QB85fHAXD0whZqipQ6qgYwBBZ7wWt3olQVtbCw99lwUARqIBbVTp2eb5Bh9MRwIKBz"
    "Ae1HFRDHXrsiCZEcYXzR0C/lHaiE/tbin4pEFq33EuftOxaooQBo4E65VRcvmNLhX//5f08cy9yRExE7RsqTMax9FQtY9UXj"
    "uFBVvaBpepBaC6Kn/kBVO2eYdPlZhcZqeXrp89QfprBT0Pr8DmM/WYB2An+EyXgdHgejD1So1KGWpaCZaWGtAVmLUvgLZyXe"
    "ClG30BRY7FGt18LyFw7k9V8eeYO//vP/cd230Ne62C6bLSSaQhn/sLdjd45OleOgdu/G7k7sYjECyS/rdMZdQTGE2E0SC82n"
    "GSfLPNWE5eyj2oFpS0Ki2vOikIg11cQsfTCIDr4+iIi+61UUWM7QM02GSiUQZ4Qvb9ujbtMUB3S31y16WCcHLJKO8m5yupuV"
    "raldzOxsJrqeiO5Au0dwqj2o3IXSDjdXzzNsdi+sTnAvkuFJRglXHEq1UMWLPVYqVsLAG53U702r9mb4Vi2ekx7LeCRcUfFN"
    "d6uynhDTFeKgXvvP7mQ7+uyOZG21MeYwjoGl164MKbspEDP2VPBzsIZbGMZe1WbnK/aNqtQHdSPdwo+iwCQGpy3US54wSjWl"
    "DJIpe++LMiS25QbD5yV6ivA6tawzpV6ylnkBO7yZLTJUMCOMhqgmEgLDk/MdloUot0vzCEGVaGoXvlPzgtgpUCNLyhEpVhxP"
    "DXk8jR24JHh8LjltPvAU85H8KRcKU7u84ruZV0JDespSwo9g30xTq2sEnouJ0Rc9tbq94DHfUpYFprtETDBsKYqp7v+Snweo"
    "Gb4wasZTynmfhg7UhLc8LPB6KzJ9puSEmFoNZDEtfIlBqam1Hww85mnJgdJ0ZBMIWz5qqneBkZ8HKBm9MErOpz7ddmWPcHV9"
    "8bjHZjE/LaUMnFIp1Klvv7Qr58bv8Xyc2sUwPudJ2gNS36AWYPyJa8jl+sEAbeMXJJnDqWhsFbl6PhqxVvfCwCnx7lTYNMTR"
    "7nQpJZvqim2KOL3Aq1My9mfdx8lQiw4ndPTI4zAirCtb92Q78ELywvZ5NF2Qvsoe2WIZxXLrNcwN64ZMmoNX4B7dZXj/a7oQ"
    "lzjCafBG4ZC/vwRDg6LHfjD103dCnWrKQAnNCBGypaxMQ1txn4ejh7M6I8amtkz1qi7y8wDLzE/JMg/Np78uPn9GUSmam1Bz"
    "rxTXTgBvS8FCcv1BKU8JGfigEEO5qfZdTKT/E/R916ver2sQdDlD9mvFsUX1gTXxxAhZhSlDKqRTzByg7GwCvJU6XlfYpKSg"
    "xaSghXY9EB6rqHWsP1fwuuAxAfNSpwNdEUAuWmL66Aw0rhMFpcB79Y0MyMi+//GqEBd2vgKe+ZR56/z+Z9hOYNJQ4w4aoBUp"
    "bIY0L39wcdeZrFYXQ8UiNFORFh1Es25pYdIdkKRR2nYY59w+DXJ9Tj1HTJR6wwwOPY4gDRF3jxDxzOL55NSR8fRp0GtafKOJ"
    "d9yt97/S4geYVIybNhGT4G6ktJM+iyNUjIHYbnHEpsXxAAz0IhQGbN6kTAyT0kFBMeKwPbGcR9BYj4zTTUXEeuqmsyp+GOob"
    "orhysI4FTfVm+lpcU3lDRr1WPk4vSCl6fOVYklI2MbugAbg7p9aoHjzu02GPQ48YZAxi7LEx1eN/8vMA0/gv7bRAsYs13B15"
    "RjQDVpRDGC+UQ7Ab31A+s6ZeDJAiuqABtH8D+9nRyyNu6DjwhI4If5AExkHiwhPByQ+SUzQnqeueJjO1EOc78g/ANb7lvsBu"
    "B9tsLziF7UZiefehGkgg5X6iKiBJI/t9FBN+UKfSsZWS2QYlxyBCPU97UOipl8w1LTHvUhn9D4h3wufxmFgrazc7SF+1xZSt"
    "dQOCfnP/I3T8KhshLzrPedQ7lZYQnftWr0jrwNBOf7lhIW9gIQ5xFHdWW6UTdnRERveJjEGDt1yMK1dTB83Tj57NWuF6LEBl"
    "H7Tr4IY0uflh3fbZZ4ryZZ/vfwXnQX2X/JoaG2XkSkK4cmuJ8e3F8awep/bHc2qEq+xTEgIBmQvRm3pHJtOIk0Io/el21RR6"
    "A2NzUmiMWL0jI51+zZ2HK5LZeSqof3SfVHJBriOTXNHieUzDTeE1FxkVvzhldw7KLnqrw+Mip5V6919RNS+oelckmyR1PXHh"
    "NBTuLixGDesErO9Zs039UdFSF3R0z5sdEVab1XXZoUyIyoyi96ayVUusBP6DlSXmJ3K5kdaCLiUoO77OqMxEFMz0ku1w9mAu"
    "YPyKWhAXy+KaqlAcsGxAtc1R3cEaKsVNqdWu0LSZeuCWTZohAMqim77T2USr0ohYwr7Hp5TqYK/TN4t5dcURJN0X6QDjEyti"
    "VjHxZ7EDg7hdCX4Qb4zt64rFKOj3iFoUUDLiQI0BVgfsgYIrlSpdgaC4SVl3X6cvF4nWLPMf0Cfb/gLqWQe9IzH1xysv94cb"
    "N1dbPeZCzv9eIoGXemYza+qNOqwNyixLqPVwIUZRzSQ5DVBbrV2hBv8yBf8LepGCP6JExOyV/a7rqz5teejX6MrT+N9BKZsj"
    "fgF/efbVyNuzlesN6ac47lDbQr8weP7Lj6g2ISev6PTDkijoC4ZfSH1/cjoYQYcodqicoUhYCorO407z1zpp8Bp7yJeC5sNc"
    "TBFBCftCTiuOGSJ8rLNxU1ZHHYbpfgJoPcF7MZDqQzLNrxKu+dVI3NR4w0is2IAn0xsrGe6NRawWnRGroVy8/wV077ItEOWn"
    "M6VrUgZ6ilEnii7MgFhCVsSv9XJQYqSoK+bEX2IqBHtBsImEIUqxJLTxFhzdEpFVKUbAFTIKBTAVLP25xKqPk3qx0LioFz4r"
    "dDUkWnk7VMySOCY+K3lb62NbedZkkOipO5Kg/itsrY+iF9cGMAHrGx1FnGzVnUpioDbOpUc1tdNtRQ2Vi6JStJgWLoRMpGpn"
    "faImiltRE8VWYQmPe5pXuyEzJPFMNNjaubEu8SIXJS45KVM9zOaTWxGXyp/RPkzjGe9ZwncUnXgl8xvEcmsJVTAN+bip5YhV"
    "/07jjso9ND31TNMnZm8B6vmpqQth5mdKGHFABrM+wgwZK1pFv6CW19QuJrRnuM265HEAQjFjc3rWf2JktLkI8fT5jmS+C5ow"
    "duSxrHZBFUcrluve0I8XpSbAxQILSEuHOubf/OevSKT7i1cP1hlhrguJgSDBgjqCzCy9iOTrqj4mBr6lARiXWTCl9uiLYSWR"
    "R0atq+eIBhsgWtgL69FzB4ZaPKPTheGoViyjk+q6FObysqxgfUAVIr9cRK5HOAxh0a4OmJfCyHg4g0DMwTr32s96S+NmjClI"
    "RV5iLARpGjlK9TRqxWkavbL3vHrV12PZGaEhyW6iwrfCemW0wuoNHCEnOd4ifxr50WUFCA/A78QPqA5RFHK4k6JQnBKxvOHM"
    "kd6h4jAI7S6PcPaKbYRoAh2iqQmOFzevRssbx7vjTyZvQAwDwTLoCoUkBOGRY+Ifpn/CUsIhIW5/wVKAcZBYXBEfD7R7jCMC"
    "swT3pZOAke9eyOnfS3DCdu9PtaaFh8QBcPbWngahlte+KD929Kn5in7u280EP37VJ5r6fwaTou3yA4aY1USdtVsN08J30GD9"
    "4Dw1WGnh92qwQ7rlLNZuwNYqxYwcBYwnq6PBOgAR113N6XmXlOGJciBSeHYa7B6vMFNYXqqwVNK49dkL91THG4pnbadmNPG7"
    "d//TaBenhCncP3SUJw4+zmbce/G+6HA54/pqjnRtMjixvk0Wm56Om1bllZ478FJ0dry0OeDxeF00Odbk2v2UXWLv7H0uSu0B"
    "j1yuZcsjcRKTIZS+0lKelEEQgV1V9z+Ok1BB65ieibqfnBkkh3iZKHlQ41ZP8darS3zPGdaiLxSEnUSPFT/DMhqHGV8Ov4ue"
    "UhPfjdfi0xtKD8+2VP0kgfBgRXGPnySY9YoAf6JdzJd+QBHXcGsZ7wqEQsfG9CdqFZ8vDyNyCeDtjM0uSU8U8ICLKngnXeqs"
    "daEJ6j1TX9xBjTXQ/R+YoSReZTb+d2tk5zpOKn4J7Y5bTMePLS3dxJnt4/3MbHNJ46jzxC29S6GonqYWOqmdoGuywgEuv4VH"
    "NbIgHnBRB53Jf7rkImWlwtu2ulufU0JZszoLUKGrmARV8ZArcoOvLDgPg212fNecl6NNM7lriRuiTnjyHfooJApu6xZiPR1V"
    "BXt3uDu4rYv7cmahpN6co9mcaR/DgrKZbg2QPgLq+8tle0Vnv71uvakoZRs43TRQ9wEKqsjYYyL7OmIbXNJhlvalaXYh1PtM"
    "m7uPTiP7VRKl4sen1LFXRBRqxbeyoVbIkatWDFOT+2NamT7/SyzIw9LNINsgJNs+a2D00W9UFXaiXvIiqBe9RvIFb2SDgdEC"
    "LcLFe0/JyrbuWyMPSmdIqr3xSOfk/EXQb45yjlphkvrGCbuvrqrDdVPf9FYYZk0zXprmvVBrfOygac9ojuh7SN2my1RgKmdo"
    "49HOYs9Vnb6j0dAzmfsQ6D189bPXgfjp09ghI33ozQ0JxRoBQxtWJlENEigZbgRkj7RHAiRHUtc4N/d2ItWXILK2T9CUGN9+"
    "rzNwNE1URNStr6DQo0H5PXc3jaubQW+8jFhjccpz4cFc8e2h2EmfFSiX93+B3w5ZUKKVJVYGuJNLUbeBQ3sWGu/54l3NcSVG"
    "i7lgKMMskNr3lUw5zXEH46B3noIGKd4Ku7REFSbm61vRTw2OLL+9v9Iq7KFv6S7sYROVVZe8gvp48/u2bl9saW0vsDRUBbff"
    "gYOO+AWa/eyKO8ukyg9oWTUcPsacYrvPL8NQ38ca2kHTIKfHWgu6vogpCcCY69VA0iKUR0HMSSo/Zmy2HoCdvg0GqD6Z5Mej"
    "AmVETv9MyQkSBlrVQP5lRvvjjm6UEF+/pw+RKY9ESBIE2e2HfpMB1QRSAZtf5cnrggXej4KrlmPMde81HodUYSbH0AdeNkD8"
    "wjeO+ukxaPWZ9j0InVSZ9YPz55ygYR2stY9/kEZ7BOvs9sX+IBsk5Vrba9NBRCc5Xf/3VZW3UVMvpFpNLyhFR8NGp+H0HnkI"
    "RByF4zeOKvPRPwencf4hNiZ1+AknVbqdQszP65lK6IfRSkZDgiq6fTq/Ig/r6RyLfvToirEr9f4T3pqle05ZU19NJgls1bt7"
    "TZ9oNSFAJhNQf2kxB68JKy/V427BOm3NGNpfi4EkI7nRwoAJ09IXtgyAkRj0bHIBm29wYY/303MHzojPfmOjt4HUcWZXw39p"
    "NW8x4bfEzj+4z6g7OWUA43lcimlQi701XNCuG3o0mNoPfcK9fJWXI+Kw8PYm3+OHLtUWp7t3BqoFXA4qPUol3l5B7LyCQv+U"
    "nuUhfHH5DJ5eldT+sH0GV0o/lZoSAYkc9z9Wecbf26avpZ8FIcLBACCpVqUAAT5/Ma/Z3iClzWbtV5HUSNW9DLqXt93htu0N"
    "JES2h4UdbBvySxy4wUV7fJmMAGfJ/c/0Ds8N7Tt20hAZXguQbwWYN1olffxatPElNzsV95jVkY/uTbGZRLrDHkchIurm96DA"
    "3hJT8Wj1uJkDv4SPzS9jrx+CI6+QC/WODDf5YUe8s8X6kFe0N7BR31z+rSoFxGKYvw91A/KqHv6BLyjSACBUC6rjIYjWoiKd"
    "cXglVEJUjhO6J/o6ok6uc1Lp6YlPGReJ1Y2RxHKebkGR49ChU2YIETbJVMWmzTWNHfgneo7zhssybdenTijNPOQzSg6eeZgz"
    "Jnclcg9+m633h4rO8UT7+gFSSMECxr4m+CgMCNIb0c6lZZ7XPiXoJW/qvilJ00GFuzan/w43EeWCFE7Qjw5/76EHJ+fbEMRP"
    "8bnwk3FNzjzPmy5Hy8OdVP3o4FloKzFCUqltk+gKEc5pNi5e1E24umyh8++wQLLDa/oTS0gsxbvgWjq76CnJk2itRyZ6qjtn"
    "ef9ztcR9Ax/hVUg7jnVmh8JpWHLy//1PqufEOhu+YaaF5YehsOI0OXTpa0mOs+E3oqtd/YmqAM3TiVWu4PNjzgceHRw2hAjb"
    "LKQfm9bMgfd6o3nEMvPnUlX6uAX/V26vwO23EksJtn0BS7n9iCUS0P/W8M38KL75Vgew7sxfa5MK2UyVtqYYtnuIwond6qEv"
    "eg6RoxBpVNouCqwya8Wj1Wrxa4cDJj1PQaJtOkaGNPaRm8KhnuQ0SVd4ODX0G2f0slDHtfPrN3kHq7kTqRfP5gHhLuwjeIvd"
    "G9e/FmOPc09XNSmwb66GUGRP7P6I643YU7ju1A4QDWZf/bp+fpg52CROofQnExRGSKxcU8OMeg9d0uYHO6QqkADEJpGjCmmZ"
    "C4CImX77xVQdozr7NbHu/yQyBPX9r5fA7l26D0BtFEgVHmt39gBtzVCHI8Ap3P5kJP+dasVjRdDdAa5Q0yU0sOeF6Skbu9Z+"
    "DbLReV9GyXu9dGA6oEGnxREuilHwWq/EI/og/OCciE9+iOyaqqoILwQa2zvpgrgz/AwdN0R0OjeERALtfWHwb7I7kDtvhUsE"
    "kxErYea/aV0BqmfiNfXNTpR3cDun9R9cV/fj3BNHIUsei2E0+ViZk9sicGDB8Fy0ze9zj1J0oV4k+D+zilbX4oinek8dNzxd"
    "How4T7yqoawkALXI95bvYSXepQIzVG4mpyBZ45O4cPPMBzMK5zd+8ZDki/XGQZJ6jBFzHCqtb55Dgj3YUptzPkkduCo6u4Ot"
    "8SKqKp845hD2XnAGZgXZD7JGqGG554Kabh/YqP13osfhPgcfuLja+V0BQOEHZpA20Q4XXgn4960X0d8LpBDmIGtFm79VQeJJ"
    "Aw22yYdhFX9xjyl8EsRwyh6U2Ku6vGnsIivj8/HXYzLED1LnNdxE9YH0DlipPoUotE67dd+JDUql1FVXhuj8bpNp52HHbJFi"
    "RjWcYtGZTBb8d3TPu+nRozDCuTq4OPvmbTr2kGsephxz4RsQWIE9AIm2f0QegaoA+O7Kwk7+7LTlwpXCZqIaGeV/Ns0GMNVI"
    "NGcQ+StgG02gM0M4gayVaILJ39Fk1i2RVxJ8JNXuYE7hkBW7hbwlAQovAE0wPIfA8F9RY3sS96QcqY0nAAM5GQ6i0kCUs+sH"
    "1jOMvuDTYl3wsiUiuWLEevhqtFqnXv1kgAWdcmNPyX0jKntflutd1tZajvEBLHJV/pDt8BzLPh/WVJ0kgPr01Ur2NgD1CRpz"
    "V5kMVsIQVEeg30dV115ei5PAU38LLju1Z80zmvcD26W6KSFDyXM4jtDCw+xtk8mvFVb91gBFmwYz8D5CL/gqu1xTmwssUS4e"
    "X5e7Ah9ejOtl7YqYFJMPQemEHa+JC8NH5MKTlcRSTy3qPyLulWF3XyrjfFffxOo5u+D/tDa4kpSS4uMZEUR1rRLHI6uZI47q"
    "Uge+VWn3mTvCA4g4nFF2FFg/pH6F2E8dmCI6W6ZQpBmsyqZcZVRcC06qHw6oj0OSeUz/mwNTLHgbjkaqmf14AtENth0Vfyz2"
    "ucITWAwNStBm1OcAPdk3aCVSfw7U72gcHnk7PPJu8s/vPCgv+DPNuW5h2VXzjk3nghoOOhVSrHI+Rh8ntomfnG2OdF7j78de"
    "VVuDe6J3mPLrA/v4eD8uCHsyrUBfRJcXLiIaj14JYnArc7nQTxrX+1QXRzVclcBLwnKtdtVqcr6wSpo9sBFb+jc54yQl0XHY"
    "8FUYx90tQ55JzkELsrHLV9DMOtt+Fl6nFQYFUWXOCxTXVbYb0mhKuj1OgyOGi/5jhh5UWEe4yidrfAhRT9TC/g1bciI2ikG/"
    "YuPbmxv7ZnPj8WAZtcXv6WjsGx2N/aGOxsQM82exyo7hCDxAt9A7xEmxzdZXGXFNDyMENopc0CgPq0VCdEW9C2+jk37Ad6nl"
    "1uPYFRm6D38cGifraExMkz71CTK2KSV5hmR9xxms+/JTtqMkcTySyaSqraddRjXuM/HSWOMJp+I1niONjnSq1fgb7lKMA6gX"
    "ulVqweOOwvMQzNw6Jys4sRXjprrwkp8HeG/x4qyn2lUCPpLZO3KO3IEOOJs4e35gnPSqYJ2CmbjHOXDNDN+oTZVoYr9+MOja"
    "YQAzt8tMkKz/ZtR1UaC5kZHxCELqZBYS0L4CD2ypK7loM0NZ0YAoz3uZxUhVMRRRTFH4pX2I0UytWwoEyGO25IHf7WMzCpDo"
    "mNKCOKEq6lbm4LkECcRCSYRsMfIOKQo/5LiJZSNWLAuANV1RYoCPyJQmI+RIW7SHRApYlz0mUOOGQSNUQ4FEG/AZGFRQNgkd"
    "ND40LBMmiHm892u18RTXkURcc8wLy9VVoTkhpowi3I/jSbUdP3jpNrZca1SAuHADLH5UW6qG10KzV023IGqr0nrV2Op11CZV"
    "sCFSQ/6cBgdK6jChn8zh4ofnajxj9ApSN66hFDdIb+i0gYEp8KoA7YHwkFJVol6XbykGdf/rWnR/WBbbZSk6HK9sbTiUpRYT"
    "YIqHfP3DsPuXxmBKhPDJ+6hNwP9nZtCgU6ixQQmXiDro0JVtm3orgPCe334crGUiGeisItui0CqzDrwUnbP0e6Az5vd/r/pj"
    "Ei77UOxgq28EfMqgh5JcAE5MhmSW1RMTm56YMQBZ0cP7YGLDBxO7+GD8+FzPlG5DT8guAYkFS4QcADmZbTdPWLKP97/uIQrX"
    "BCbXMDdNIIeM0YrkxJky74MNbaXfZjzVcnsG7NlYjsBUCbuRHR5vZJtYOdjXBj58gDI24pOMhPv//4ObqfNRAQA="
)

if not RUTA_DATOS.exists():
    RUTA_DATOS.parent.mkdir(parents=True, exist_ok=True)
    RUTA_DATOS.write_bytes(gzip.decompress(base64.b64decode(_BLOB)))
    print(f"Corpus escrito en {RUTA_DATOS} (copia embebida).")
else:
    print(f"Se usará el corpus existente en {RUTA_DATOS}.")

sha_real = hashlib.sha256(RUTA_DATOS.read_bytes()).hexdigest()
print("SHA-256:", sha_real[:16], "…")
print("Coincide con la versión embebida:", sha_real == SHA_ESPERADO)

### 5.2 · Carga y particiones

Las particiones vienen **fijadas en el archivo** (campo `split`), no se
calculan aquí. Es una decisión deliberada: si cada notebook hiciera su propio
`train_test_split`, cuatro arquitecturas estarían evaluándose sobre conjuntos
distintos y las métricas no serían comparables entre sí.

La partición es estratificada por categoría (12 entrenamiento + 3 validación
por clase), generada con semilla 42.

In [ ]:
import json
from collections import Counter

registros = [json.loads(l) for l in RUTA_DATOS.read_text(encoding="utf-8").splitlines()]

train = [r for r in registros if r["split"] == "train"]
val   = [r for r in registros if r["split"] == "validation"]
demo  = [r for r in registros if r["es_demo"]]

CATEGORIAS = [
    "suma", "resta", "multiplicacion", "division", "operaciones_combinadas",
    "potencias_raices", "fracciones", "porcentajes", "ecuaciones",
    "geometria", "estadistica_probabilidad",
]
CAT2ID = {c: i for i, c in enumerate(CATEGORIAS)}
ID2CAT = {i: c for c, i in CAT2ID.items()}

print(f"Total: {len(registros)}  |  train: {len(train)}  |  validación: {len(val)}")
print(f"Ejemplos de demostración (todos en validación): {[d['id'] for d in demo]}")
print()
print("Distribución por categoría (train / val):")
ctr, cva = Counter(r["categoria"] for r in train), Counter(r["categoria"] for r in val)
for c in CATEGORIAS:
    print(f"  {c:26s} {ctr[c]:3d} / {cva[c]:2d}")
print()
print("Ejemplo completo:")
print(json.dumps(train[0], ensure_ascii=False, indent=2))

Un ejemplo del corpus se ve así:

```
entrada : "María tiene 48 caramelos y quiere repartirlos por igual entre 6
           amigos. ¿Cuántos caramelos recibirá cada amigo?"
salida  : "Paso 1: Repartir en partes iguales es dividir.
           Paso 2: 48 ÷ 6 = 8.
           Respuesta final: 8 caramelos"
valor   : "8"
```

El campo `valor` es la clave de toda la evaluación automática: es la respuesta
en forma canónica (sin unidades ni texto). Comparar `valor` contra lo que el
modelo escribe después de `Respuesta final:` nos da una métrica objetiva de
**si el modelo resolvió bien el problema**, independiente de cómo lo redactó.

In [ ]:
textos = [r["entrada"] + " " + r["salida"] for r in registros]

filas_tok = []
for nombre, tok in tokenizadores.items():
    n_tokens, n_unk, n_palabras = 0, 0, 0
    for t in textos:
        ids = tok(t, add_special_tokens=False)["input_ids"]
        n_tokens += len(ids)
        n_unk += sum(1 for i in ids if i == tok.unk_token_id)
        n_palabras += len(t.split())
    filas_tok.append({
        "Tokenizador": nombre,
        "Vocabulario": tok.vocab_size,
        "Tokens/palabra": round(n_tokens / n_palabras, 3),
        "Tokens/ejemplo": round(n_tokens / len(textos), 1),
        "Tasa <unk>": f"{n_unk / n_tokens:.3%}",
    })

df_tok = pd.DataFrame(filas_tok)
print(df_tok.to_string(index=False))

In [ ]:
frases = [
    "¿Cuántos caramelos recibirá cada amigo?",
    "864 ÷ 12 = 72",
    "El área del círculo es π × r²",
    "√144 + 8 = 20",
]

for frase in frases:
    print("=" * 78)
    print(f"«{frase}»")
    for nombre, tok in tokenizadores.items():
        piezas = tok.tokenize(frase)
        ids = tok(frase, add_special_tokens=False)["input_ids"]
        n_unk = sum(1 for i in ids if i == tok.unk_token_id)
        marca = f"  [{n_unk} <unk>]" if n_unk else ""
        print(f"  {nombre:26s} ({len(piezas):2d}){marca}")
        print(f"      {' | '.join(piezas)}")
print("=" * 78)

### Cómo leer esta comparación

**Qwen (BPE, 151k)** — El vocabulario más grande de los tres, entrenado
multilingüe. Suele dar la fertilidad más baja en español y representar los
símbolos matemáticos sin recurrir a `<unk>`. El costo es un embedding de
entrada enorme: 151k × 1536 son ~230M de parámetros solo en la tabla de
embeddings, más que el modelo FLAN-T5 completo.

**BETO (WordPiece, 31k)** — Vocabulario pequeño pero **enteramente dedicado al
español**. Es la demostración de que el tamaño no es lo que importa: importa la
correspondencia entre el vocabulario y el idioma de los datos. En cortes
morfológicos ("estudiantes", "dividimos") suele hacerlo mejor que tokenizadores
multilingües mucho mayores.

**FLAN-T5 (SentencePiece, 32k)** — Del mismo tamaño que BETO pero entrenado
sobre inglés. Es donde aparecen los `<unk>` en `¿`, `÷`, `√`, `²`. El notebook 4
mide cuánto y lo mitiga normalizando. La lección práctica: un vocabulario del
mismo tamaño puede ser excelente o inservible según el idioma para el que se
construyó.

**Consecuencia para el proyecto.** Si el tutor debe operar en español, el
tokenizador es un criterio de selección de primer orden, al mismo nivel que el
número de parámetros. Un modelo mayor con mal tokenizador puede rendir peor que
uno menor bien ajustado al idioma.

## 6 · Análisis cualitativo II — Fine-tuning

Tres ejes: estabilidad, sensibilidad al dataset pequeño y convergencia.

In [ ]:
filas_ft = []
for clave, nombre in [("qwen", "Qwen2.5-1.5B"), ("bert", "BETO"), ("flan_t5", "FLAN-T5-base")]:
    if clave not in R:
        continue
    ent = g(clave, "parametros_entrenables")
    tot = g(clave, "parametros_totales")
    filas_ft.append({
        "Modelo": nombre,
        "Método": g(clave, "metodo", default="—"),
        "Entrenables": f"{ent/1e6:.2f} M" if ent else "—",
        "% del total": f"{100*ent/tot:.2f}%" if ent and tot else "—",
        "LR": g(clave, "learning_rate", default="—"),
        "Épocas": g(clave, "epocas", default="—"),
        "Mejor época": g(clave, "mejor_epoca", default="—"),
        "Tiempo (s)": f"{g(clave,'tiempo_entrenamiento_s'):.0f}" if g(clave, "tiempo_entrenamiento_s") else "—",
    })

df_ft = pd.DataFrame(filas_ft)
print(df_ft.to_string(index=False))
print()
print("La columna 'Mejor época' es la más informativa: indica cuándo dejó de")
print("mejorar la validación. Una mejor época muy temprana respecto al total")
print("significa que el modelo agotó lo aprendible del corpus enseguida.")

In [ ]:
for clave, nombre, total in [("qwen", "Qwen", g("qwen", "epocas")),
                             ("bert", "BETO", g("bert", "epocas")),
                             ("flan_t5", "FLAN-T5", g("flan_t5", "epocas"))]:
    mejor = g(clave, "mejor_epoca")
    if mejor and total:
        frac = mejor / total
        if frac < 0.35:
            lectura = "convergencia MUY TEMPRANA -> el corpus es el límite, no el entrenamiento"
        elif frac < 0.75:
            lectura = "convergencia equilibrada -> presupuesto de épocas razonable"
        else:
            lectura = "seguía mejorando al final -> probablemente convenga entrenar más"
        print(f"{nombre:10s} mejor época {mejor:.0f} de {total:.0f} ({frac:.0%})  ->  {lectura}")

### Estabilidad

| Modelo | Comportamiento observado |
|---|---|
| **Qwen + LoRA** | Estable **siempre que** los parámetros entrenables estén en fp32. Con los pesos LoRA en fp16 el gradiente se subdesborda y la pérdida se va a NaN en pocos pasos. Es el fallo más común del notebook 1 y por eso el casting es explícito allí. |
| **BETO completo** | El más estable de los tres. Fine-tuning completo de un encoder con `lr=3e-5` es una receta madura y sin sorpresas. |
| **FLAN-T5 + LoRA** | Estable en fp32; **inutilizable en fp16**. T5 fue preentrenado en bfloat16 y sus activaciones desbordan el rango de fp16. Como la T4 no tiene bf16, fp32 es la única opción. |

Conclusión transferible: la estabilidad depende menos de la arquitectura que de
la **interacción entre la precisión numérica y el preentrenamiento del modelo**.
Es una fuente de fallos que no aparece en los tutoriales y consume horas de
depuración.

### Sensibilidad al dataset pequeño

Ordenados de menos a más sensible:

1. **BETO** — Es el que mejor tolera 132 ejemplos. Solo tiene que aprender una
   frontera de decisión sobre representaciones que ya son buenas; el
   preentrenamiento hace casi todo el trabajo.
2. **FLAN-T5** — Tiene que aprender un formato de salida completo (categoría +
   pasos + respuesta). Más que aprender, sobre todo tiene que aprender a
   *imitar*, y para eso 132 ejemplos alcanzan razonablemente.
3. **Qwen** — El más propenso a memorizar. 1.5B de parámetros sobre 132
   ejemplos aprenden el corpus de memoria si se lo permiten. Por eso se evalúa
   por época y se recupera el mejor punto en lugar del último.

### Convergencia

- **BERT** converge en pocas épocas y luego se aplana. El aplanamiento no es un
  fallo: es la señal de que agotó la información disponible.
- **Qwen con LoRA** baja rápido las primeras épocas —está aprendiendo el
  formato— y después mucho más lento, cuando le tocaría aprender a calcular,
  que es lo difícil.
- **FLAN-T5** desciende de forma más gradual y sostenida: tiene menos capacidad
  de memorización, así que el sobreajuste tarda más en aparecer.

## 7 · Análisis cualitativo III — Curva de aprendizaje

Tres preguntas distintas que conviene no mezclar.

### ¿Cuál aprende más rápido?

**BETO**, sin discusión, en las tres acepciones de "rápido":

- *En tiempo de reloj*: segundos por época contra minutos.
- *En número de ejemplos*: alcanza su meseta antes que los demás.
- *En épocas*: su mejor época suele estar en el primer tercio del presupuesto.

La razón es que su tarea es la más fácil. Elegir entre 11 opciones es
incomparablemente más simple que generar una secuencia correcta de 40 tokens
donde un solo dígito equivocado invalida la respuesta.

### ¿Cuál requiere más datos?

**Qwen**, y con diferencia. La razón es la estructura del espacio de salida:

| Modelo | Espacio de salida | Ejemplos necesarios |
|---|---|---|
| BETO | 11 clases | Decenas por clase bastan para una frontera decente |
| FLAN-T5 | Secuencias cortas y muy estructuradas | Cientos |
| Qwen | Secuencias libres | Miles, para que la mejora sea de razonamiento y no solo de formato |

Con 132 ejemplos, Qwen aprende **formato**. Para aprender **aritmética**
harían falta uno o dos órdenes de magnitud más, o un enfoque distinto
(herramientas externas de cálculo, verificación simbólica).

### ¿Cuál generaliza mejor?

Depende de a qué se le llame generalizar, y merece la pena separarlo:

- **A ejemplos nuevos de categorías conocidas**: BETO. Es lo que mide su
  accuracy en validación.
- **A tipos de problema no vistos**: Qwen. Un decoder grande arrastra
  conocimiento matemático de su preentrenamiento y puede intentar problemas
  fuera de nuestras 11 categorías. BETO no puede: su espacio de clases es
  cerrado por diseño.
- **Al formato**: FLAN-T5 y Qwen, ambos aprenden la estructura de salida con
  facilidad.

**El punto que no hay que perder de vista:** con 33 ejemplos de validación,
ninguna diferencia menor a ~10 puntos porcentuales es estadísticamente
distinguible. Un solo ejemplo vale 3 puntos. Las conclusiones de esta sección
son *direccionales* y están sostenidas por el razonamiento arquitectónico tanto
como por los números. Presentarlas como mediciones definitivas sería un error
metodológico, y explicitarlo es parte del rigor del trabajo.

In [ ]:
print("Recordatorio de tamaño muestral")
print("=" * 60)
n = list(n_vals.values())[0] if n_vals else 33
print(f"Ejemplos de validación: {n}")
print(f"Un ejemplo vale: {1/n:.1%}")
print(f"Intervalo de confianza aproximado al 95% para p=0.5: ±{1.96*np.sqrt(0.25/n):.1%}")
print("=" * 60)
print("Cualquier diferencia entre modelos menor a ese margen NO es concluyente.")

## 8 · Recomendación final

### La pregunta

¿Qué arquitectura desplegaría en producción un tutor de matemáticas en español?

### La respuesta corta

**Qwen2.5 con LoRA como generador, con el clasificador BERT como componente de
enrutamiento y observabilidad — es decir, la arquitectura del notebook 3 — pero
solo si el pipeline demuestra aporte medible. Si B ≈ A en sus resultados,
desplieguen Qwen solo y conserven BERT como instrumento de monitorización.**

### La justificación

**1. La tarea es generativa y eso descarta opciones.**
Un tutor tiene que explicar. BERT queda fuera como sistema completo: es un
componente excelente de una arquitectura, no una arquitectura.

**2. Entre los dos generadores, Qwen tiene la ventaja estructural.**
- Su tokenizador maneja el español y la notación matemática sin `<unk>`;
  FLAN-T5 necesita normalización previa que degrada la salida.
- Su preentrenamiento matemático es más fuerte.
- Escala hacia arriba (3B, 7B, 14B) sin cambiar el código, lo que da un camino
  claro de mejora.
- El precio es el costo de inferencia, que es real pero gestionable.

**3. El clasificador se justifica aunque no mejore la exactitud.**
Este es el punto contraintuitivo y el más importante de la recomendación. En
producción, un enrutador de 110M que cuesta el ~2% de la latencia total aporta
tres cosas que el modelo generativo no puede dar:

- **Observabilidad**: saber qué tipos de problema llegan y en cuáles falla el
  sistema, sin leer texto generado a mano.
- **Control**: rechazar preguntas fuera de dominio, o enrutar `operaciones_
  combinadas` a una calculadora simbólica en lugar de a un LLM, que es la vía
  realista para arreglar la aritmética.
- **Degradación elegante**: con confianza baja, pedir aclaración en lugar de
  inventar.

Ninguna de las tres aparece en las métricas de este laboratorio, y las tres
deciden si un sistema es operable.

**4. Por qué no FLAN-T5, pese a su elegancia conceptual.**
Un modelo único es preferible a dos, y hacer clasificación y generación con una
sola pérdida es la solución limpia. Pero `flan-t5-base` arrastra un tokenizador
inglés y un preentrenamiento matemático más débil. La recomendación cambiaría
si se usara **mT5** o **flan-t5-large**: entonces la comparación habría que
rehacerla, y ese es el experimento natural que sigue a este laboratorio.

### La condición indispensable

Ninguna de las cuatro arquitecturas, con 132 ejemplos, produce un tutor
confiable. La exactitud aritmética es el cuello de botella y no se resuelve con
arquitectura. Antes de desplegar cualquier cosa haría falta:

1. **Más datos** — el corpus está preparado para escalar: basta reemplazar
   `data/math_tutor_dataset.jsonl` y re-ejecutar. Sin cambiar una línea de
   código.
2. **Verificación externa del cálculo** — que el modelo genere el procedimiento
   y una herramienta simbólica (SymPy) valide la aritmética. Es el enfoque que
   usan los sistemas de producción serios, y encaja de forma natural en la
   arquitectura modular del notebook 3.
3. **Evaluación humana** — un procedimiento pedagógicamente correcto no es lo
   mismo que un número correcto, y ninguna de nuestras métricas mide lo primero.

In [ ]:
resumen = {
    "notebooks_disponibles": list(R),
    "n_validacion": n_vals,
    "generacion_exactitud": {
        "qwen": g("qwen", "finetuned", "exactitud"),
        "pipeline": g("pipeline_qwen_bert", "config_B_pipeline", "exactitud"),
        "flan_t5": g("flan_t5", "finetuned", "exactitud"),
    },
    "clasificacion_accuracy": {
        "bert": g("bert", "finetuned", "accuracy"),
        "bert_baseline_tfidf": g("bert", "baseline_tfidf", "accuracy"),
        "flan_t5_implicita": g("flan_t5", "clasificacion_finetuned", "accuracy"),
    },
    "tokenizadores": filas_tok,
    "tiempos_entrenamiento_s": {
        k: g(k, "tiempo_entrenamiento_s") for k in ("qwen", "bert", "flan_t5")
    },
}

Path("resultados").mkdir(exist_ok=True)
Path("resultados/comparacion_final.json").write_text(
    json.dumps(resumen, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print(json.dumps(resumen, ensure_ascii=False, indent=2, default=str))
print("\nGuardado en resultados/comparacion_final.json")

## 9 · Cierre

Lo que este laboratorio deja, más allá de los números:

1. **Un protocolo experimental correcto.** Mismos datos, mismas particiones,
   mismas métricas, baseline antes de tocar nada. Sin eso, cuatro notebooks
   habrían sido cuatro anécdotas.
2. **Una respuesta arquitectónica fundamentada.** No "Qwen es mejor", sino
   "para esta tarea, en este idioma, con este presupuesto, un decoder con
   enrutador es la elección defendible, y estas son las condiciones bajo las
   que cambiaría".
3. **Los límites, dichos en voz alta.** 33 ejemplos de validación no permiten
   distinguir diferencias pequeñas. El corpus limita más que la arquitectura.
   La aritmética no se arregla con fine-tuning. Reconocerlo no debilita el
   trabajo: es lo que lo hace un experimento y no una demostración.